# EDEL Exploration
Quick notebook for testing the pipeline with an inline default RUN_CONFIG.

In [ ]:
RUN_CONFIG = {
    "processing_mode": "batch",
    "embedding_mode": "aspects",
    "data": {
        "provider": {
            "type": "openalex",
            "topic_id": "T10102",
            "topic_name": "Scientometrics",
            "region": None,
            "params": {"n_documents": 300, "avg_length": 150},
        },
        "transforms": [{"type": "shuffle_words"}],
    },
    "structured_abstracts": {"provider": "openai", "model": "gpt-5-mini", "min_sentences": 4, "min_tokens": 80},
    "embedding": {"mode": "multi", "provider": "openai", "model": "text-embedding-ada-002", "n_dimensions": 1536, "batch_size": 5000},
    "dimensionality_reduction": {"method": "diffusion", "n_neighbors": 15, "random_state": 0, "min_dist": 0.1, "metric": "cosine"},
    "vector_field": {"method": "diffusion", "grid_size": 25, "min_count": 3, "smooth_sigma": 1.0, "compute_divergence": True, "compute_magnitude": True},
    "clustering": {
        "domain": {"source": "proj_p", "algorithm": "hdbscan", "params": {"min_cluster_size": 15}},
        "field": {"source": "field", "algorithm": "hdbscan", "params": {"min_cluster_size": 10}},
        "style": {"source": "features", "algorithm": "gmm", "params": {"n_components": 4}},
        "operator": {"source": "operators", "algorithm": "hdbscan", "params": {"min_cluster_size": 20}},
    },
    "labeling": {
        "provider": "openai",
        "model": "gpt-5-mini",
        "text_column": "abstract_text",
        "topic": None,
        "language": "en",
        "axis": {"enabled": True, "projection": "diffusion", "n_samples": 5},
        "clusters": {"enabled": True, "cluster_keys": ["domain", "style", "operator", "field"], "n_samples": 5},
    },
    "landscape": {
        "metric": "cited_by_count",
        "log_scale": True,
        "grid": {"num_bins": 50, "sigma": 1.5},
        "scale": 12.0,
        "color_cluster": "cluster_domain",
        "style_cluster": "cluster_style",
        "field": {"enabled": True, "type": "total", "step": 2, "scale": 0.07, "width": 1},
    },
}


In [ ]:
from edel.pipeline.run import run_pipeline
from edel.viz.contour import make_contour_figure
from edel.viz.field import add_vector_field_annotations

artifacts = run_pipeline(RUN_CONFIG, base_path='artifacts')
fig = make_contour_figure(
    xi=artifacts['grid']['xi'],
    yi=artifacts['grid']['yi'],
    grid_smooth=artifacts['grid']['grid_smooth'],
    z_label=artifacts['height']['label'],
    title='Notebook contour',
    x_label='dim 1',
    y_label='dim 2',
)
fig = add_vector_field_annotations(fig, artifacts.get('field_vectors'))
fig
